# NQAI Voice — Clean VoxCPM2 LoRA Fine-Tune (Deepgram → Manifest → Train)

Bu notebook tek temiz akış içindir: Drive'a saf ses/video dosyalarını koy, Deepgram transcript + segment üretir, VoxCPM2 LoRA fine-tune çalışır.

**Varsayılan run:** `neeko-proto-v0`

**Girdi:** `source_raw/` altında bize ait/izinli ses veya video dosyaları.

**Çıktı:** `manifest_raw.jsonl`, train/val/test split, LoRA checkpoint, dinleme çıktıları.

**Kırmızı çizgi:** ElevenLabs/Cartesia/YouTube/izinsiz kişi çıktısı production training data olarak kullanılmaz.

## 0. Drive Dosya Düzeni

Drive'da sadece şu klasörü hazırla:

```text
MyDrive/nqai-voice/finetune/neeko-proto-v0/
  source_raw/
    neeko_01.wav
    neeko_02.m4a
    neeko_03.mp4
```

Notebook şunları kendi oluşturur:

```text
source_audio/          # 16 kHz mono kaynak WAV
deepgram/              # Deepgram JSON cache
audio/                 # kısa eğitim clipleri
manifest_raw.jsonl
segments_review.csv    # Deepgram segmentlerini hızlı kontrol için
splits/
conf/
checkpoints/lora/
outputs/
```

## 1. GPU Kontrolü

In [ ]:
!python -V
!nvidia-smi

import torch

assert torch.cuda.is_available(), "GPU yok. Colab Runtime -> Change runtime type -> GPU seç."
torch.set_float32_matmul_precision("high")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} · VRAM: {vram_gb:.1f} GB")

if vram_gb < 18:
    print("UYARI: VoxCPM2 LoRA için bu GPU sınırda. OOM olursa batch düşürülecek.")

## 2. Kurulum

In [ ]:
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

run("pip install -q --upgrade pip")
run("pip install -q voxcpm soundfile librosa tensorboard tensorboardX argbind safetensors transformers accelerate huggingface_hub pyyaml requests")

VOX_REPO = Path("/content/VoxCPM")
if VOX_REPO.exists():
    run("rm -rf /content/VoxCPM")
run("git clone --depth 1 https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM")

if (VOX_REPO / "requirements.txt").exists():
    run("pip install -q -r requirements.txt", cwd=str(VOX_REPO))
run("pip install -q -e .", cwd=str(VOX_REPO))

for candidate in [VOX_REPO / "src", VOX_REPO]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

print("VoxCPM repo ready:", VOX_REPO)

## 3. Drive ve Run Ayarları

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

VOICE_ID = "neeko-proto-v0"
PROJECT_DIR = Path(f"/content/drive/MyDrive/nqai-voice/finetune/{VOICE_ID}")

SOURCE_RAW_DIR = PROJECT_DIR / "source_raw"
SOURCE_AUDIO_DIR = PROJECT_DIR / "source_audio"
DEEPGRAM_DIR = PROJECT_DIR / "deepgram"
AUDIO_DIR = PROJECT_DIR / "audio"
RAW_MANIFEST = PROJECT_DIR / "manifest_raw.jsonl"
SEGMENTS_REVIEW_CSV = PROJECT_DIR / "segments_review.csv"
SPLIT_DIR = PROJECT_DIR / "splits"
CONF_DIR = PROJECT_DIR / "conf"
CKPT_DIR = PROJECT_DIR / "checkpoints" / "lora"
LOG_DIR = PROJECT_DIR / "logs" / "lora"
OUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = Path("/content/drive/MyDrive/nqai-voice/models/VoxCPM2")

for path in [PROJECT_DIR, SOURCE_RAW_DIR, SOURCE_AUDIO_DIR, DEEPGRAM_DIR, AUDIO_DIR, SPLIT_DIR, CONF_DIR, CKPT_DIR, LOG_DIR, OUT_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("VOICE_ID:", VOICE_ID)
print("PROJECT_DIR:", PROJECT_DIR)
print("SOURCE_RAW_DIR:", SOURCE_RAW_DIR)
print("RAW_MANIFEST:", RAW_MANIFEST)
print("MODEL_DIR:", MODEL_DIR)

## 4. VoxCPM2 Model Ağırlıkları

In [13]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="openbmb/VoxCPM2",
    local_dir=str(MODEL_DIR),
)
print("MODEL_DIR:", MODEL_DIR)
print("Files:", sorted(path.name for path in MODEL_DIR.iterdir())[:20])

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

MODEL_DIR: /content/drive/MyDrive/nqai-voice/models/VoxCPM2
Files: ['.cache', '.gitattributes', 'README.md', 'audiovae.pth', 'config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenization_voxcpm2.py', 'tokenizer.json', 'tokenizer_config.json']


## 4B. Deepgram API Key

Bu hücre key'i ayrı ister. Colab prompt görünmezse hücre altında küçük input alanı oluşur; oraya yapıştırıp Enter'a bas.


In [14]:

DEEPGRAM_API_KEY ="79e47402efcb69cddc14914c7e7cde1e45e736cf"
if not DEEPGRAM_API_KEY:
    print("Deepgram API key boş")
else:
    print("Deepgram key alındı")


Deepgram key alındı


## 5. Deepgram → Clip WAV → Manifest

Bu hücre `source_raw/` altındaki dosyaları işler. Deepgram API key gizli girilir, notebook'a yazılmaz.

Ayarlar:
- `UTT_SPLIT`: Deepgram utterance bölme hassasiyeti. Uzun segment çıkarsa `0.45` deneyebilirsin.
- `MAX_SEGMENT_SECONDS`: VoxCPM training için 30 sn üstünü elemek daha güvenli.
- Transcript cache varsa tekrar Deepgram çağırmaz; yeniden transcribe için JSON cache dosyasını sil.

In [15]:
import csv
import json
import re
import subprocess
from pathlib import Path

import requests

DEEPGRAM_MODEL = "nova-3"
DEEPGRAM_LANGUAGE = "tr"
UTT_SPLIT = "0.8"
MIN_SEGMENT_SECONDS = 1.0
MAX_SEGMENT_SECONDS = 30.0
PAD_START_SECONDS = 0.10
PAD_END_SECONDS = 0.20
SUPPORTED_SOURCE_SUFFIXES = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".mp4", ".mov", ".webm"}

source_files = sorted(
    path for path in SOURCE_RAW_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_SOURCE_SUFFIXES
)
assert source_files, f"Kaynak dosya yok: {SOURCE_RAW_DIR}"

try:
    DEEPGRAM_API_KEY
except NameError:
    import getpass
    DEEPGRAM_API_KEY = getpass.getpass("Deepgram API key: ").strip()
assert DEEPGRAM_API_KEY.strip(), "Deepgram API key boş"


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def ffmpeg_to_wav(src: Path, dst: Path) -> None:
    subprocess.run([
        "ffmpeg", "-y",
        "-i", str(src),
        "-ac", "1",
        "-ar", "16000",
        "-vn",
        str(dst),
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def transcribe_deepgram(wav_path: Path) -> dict:
    url = "https://api.deepgram.com/v1/listen"
    params = {
        "model": DEEPGRAM_MODEL,
        "language": DEEPGRAM_LANGUAGE,
        "smart_format": "true",
        "punctuate": "true",
        "utterances": "true",
        "utt_split": UTT_SPLIT,
    }
    headers = {
        "Authorization": f"Token {DEEPGRAM_API_KEY}",
        "Content-Type": "audio/wav",
    }
    with wav_path.open("rb") as f:
        resp = requests.post(url, params=params, headers=headers, data=f, timeout=1800)
    resp.raise_for_status()
    return resp.json()

manifest_rows = []
review_rows = []
skipped = []

for file_index, src_path in enumerate(source_files, 1):
    print(f"\n[{file_index}/{len(source_files)}] source: {src_path.name}")
    source_wav = SOURCE_AUDIO_DIR / f"{src_path.stem}_16k.wav"
    ffmpeg_to_wav(src_path, source_wav)
    print("  wav:", source_wav)

    transcript_json_path = DEEPGRAM_DIR / f"{src_path.stem}.deepgram.json"
    if transcript_json_path.exists():
        dg = json.loads(transcript_json_path.read_text(encoding="utf-8"))
        print("  transcript cache:", transcript_json_path)
    else:
        dg = transcribe_deepgram(source_wav)
        transcript_json_path.write_text(json.dumps(dg, ensure_ascii=False, indent=2), encoding="utf-8")
        print("  transcript saved:", transcript_json_path)

    utterances = dg.get("results", {}).get("utterances", [])
    print("  utterances:", len(utterances))

    for utt_index, utt in enumerate(utterances, 1):
        text = clean_text(utt.get("transcript", ""))
        start = max(0.0, float(utt.get("start", 0.0)) - PAD_START_SECONDS)
        end = float(utt.get("end", 0.0)) + PAD_END_SECONDS
        duration = end - start
        review_rows.append({
            "source_file": src_path.name,
            "utterance_index": utt_index,
            "start": round(start, 3),
            "end": round(end, 3),
            "duration": round(duration, 3),
            "text": text,
        })

        if not text:
            skipped.append((src_path.name, utt_index, "empty_text", round(duration, 2)))
            continue
        if duration < MIN_SEGMENT_SECONDS:
            skipped.append((src_path.name, utt_index, "too_short", round(duration, 2), text[:60]))
            continue
        if duration > MAX_SEGMENT_SECONDS:
            skipped.append((src_path.name, utt_index, "too_long", round(duration, 2), text[:60]))
            continue

        out_name = f"{len(manifest_rows) + 1:04d}.wav"
        out_path = AUDIO_DIR / out_name
        subprocess.run([
            "ffmpeg", "-y",
            "-i", str(source_wav),
            "-ss", f"{start:.3f}",
            "-to", f"{end:.3f}",
            "-ac", "1",
            "-ar", "16000",
            str(out_path),
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        manifest_rows.append({
            "audio": str(out_path.relative_to(PROJECT_DIR)),
            "text": text,
            "speaker_id": VOICE_ID,
            "source_file": src_path.name,
        })

assert manifest_rows, "Deepgram pipeline manifest üretmedi"

with RAW_MANIFEST.open("w", encoding="utf-8") as f:
    for row in manifest_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with SEGMENTS_REVIEW_CSV.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["source_file", "utterance_index", "start", "end", "duration", "text"])
    writer.writeheader()
    writer.writerows(review_rows)

print("\nmanifest:", RAW_MANIFEST)
print("segments_review_csv:", SEGMENTS_REVIEW_CSV)
print("clips:", len(manifest_rows))
print("skipped:", len(skipped))
print("skipped first 10:", skipped[:10])
print("first rows:")
for row in manifest_rows[:5]:
    print(row)


[1/10] source: 01-uyanis.mp3
  wav: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/source_audio/01-uyanis_16k.wav
  transcript saved: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/deepgram/01-uyanis.deepgram.json
  utterances: 44

[2/10] source: 02-gece-korkusu.mp3
  wav: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/source_audio/02-gece-korkusu_16k.wav
  transcript saved: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/deepgram/02-gece-korkusu.deepgram.json
  utterances: 57

[3/10] source: 03-sevinc.mp3
  wav: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/source_audio/03-sevinc_16k.wav
  transcript saved: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/deepgram/03-sevinc.deepgram.json
  utterances: 57

[4/10] source: 04-merak.mp3
  wav: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/source_audio/04-merak_16k.wav
  transcript saved: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/deepgram/04-merak.

## 6. Manifest Validate

In [16]:
import json
import random
from collections import Counter

import soundfile as sf

MIN_SECONDS = 1.0
IDEAL_MIN_SECONDS = 3.0
MAX_SECONDS = 30.0

assert RAW_MANIFEST.exists(), f"Manifest bulunamadı: {RAW_MANIFEST}"

def resolve_audio_path(raw_audio):
    p = Path(raw_audio)
    if p.is_absolute():
        return p
    if (PROJECT_DIR / p).exists():
        return PROJECT_DIR / p
    return AUDIO_DIR / p

records = []
warnings = []
with RAW_MANIFEST.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        audio_path = resolve_audio_path(item["audio"])
        text = str(item.get("text", "")).strip()
        if not audio_path.exists():
            raise FileNotFoundError(f"Line {line_no}: audio yok: {audio_path}")
        if not text:
            raise ValueError(f"Line {line_no}: text boş")
        info = sf.info(str(audio_path))
        duration = float(info.frames / info.samplerate)
        if duration < MIN_SECONDS:
            raise ValueError(f"Line {line_no}: çok kısa ({duration:.2f}s): {audio_path}")
        if duration > MAX_SECONDS:
            raise ValueError(f"Line {line_no}: çok uzun ({duration:.2f}s): {audio_path}")
        if duration < IDEAL_MIN_SECONDS:
            warnings.append(f"Line {line_no}: idealden kısa ({duration:.2f}s): {audio_path.name}")
        records.append({
            "audio": str(audio_path),
            "text": text,
            "duration": round(duration, 3),
            "speaker_id": item.get("speaker_id", VOICE_ID),
        })

assert records, "Manifest boş."
durations = [record["duration"] for record in records]
print(f"OK: {len(records)} clip · toplam {sum(durations)/60:.1f} dk · ortalama {sum(durations)/len(durations):.2f}s")
print("Speaker dağılımı:", Counter(record["speaker_id"] for record in records))
if warnings[:10]:
    print("\nUyarılar ilk 10:")
    for warning in warnings[:10]:
        print("-", warning)

OK: 575 clip · toplam 26.4 dk · ortalama 2.76s
Speaker dağılımı: Counter({'neeko-proto-v0': 575})

Uyarılar ilk 10:
- Line 1: idealden kısa (2.84s): 0001.wav
- Line 2: idealden kısa (1.66s): 0002.wav
- Line 3: idealden kısa (1.58s): 0003.wav
- Line 4: idealden kısa (1.50s): 0004.wav
- Line 5: idealden kısa (1.50s): 0005.wav
- Line 6: idealden kısa (2.94s): 0006.wav
- Line 7: idealden kısa (1.66s): 0007.wav
- Line 8: idealden kısa (1.26s): 0008.wav
- Line 9: idealden kısa (2.14s): 0009.wav
- Line 13: idealden kısa (1.90s): 0013.wav


## 7. Train / Val / Test Split

In [17]:
SEED = 20260524
VAL_RATIO = 0.10
TEST_RATIO = 0.10
REF_AUDIO_RATIO = 0.40

rng = random.Random(SEED)
records_shuffled = records[:]
rng.shuffle(records_shuffled)

n = len(records_shuffled)
n_test = max(1, int(n * TEST_RATIO)) if n >= 10 else 0
n_val = max(1, int(n * VAL_RATIO)) if n >= 10 else 0

test_records = records_shuffled[:n_test]
val_records = records_shuffled[n_test:n_test + n_val]
train_records = records_shuffled[n_test + n_val:]

by_speaker = {}
for record in train_records:
    by_speaker.setdefault(record["speaker_id"], []).append(record)

def with_ref_audio(split_records, enable_ref=False):
    out = []
    for record in split_records:
        item = {
            "audio": record["audio"],
            "text": record["text"],
            "duration": record["duration"],
            "dataset_id": 0,
        }
        if enable_ref and rng.random() < REF_AUDIO_RATIO:
            candidates = [candidate for candidate in by_speaker.get(record["speaker_id"], []) if candidate["audio"] != record["audio"]]
            if candidates:
                item["ref_audio"] = rng.choice(candidates)["audio"]
        out.append(item)
    return out

train_jsonl = with_ref_audio(train_records, enable_ref=True)
val_jsonl = with_ref_audio(val_records, enable_ref=False)
test_jsonl = with_ref_audio(test_records, enable_ref=False)

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

TRAIN_MANIFEST = SPLIT_DIR / "train.jsonl"
VAL_MANIFEST = SPLIT_DIR / "val.jsonl"
TEST_MANIFEST = SPLIT_DIR / "test.jsonl"

write_jsonl(TRAIN_MANIFEST, train_jsonl)
write_jsonl(VAL_MANIFEST, val_jsonl)
write_jsonl(TEST_MANIFEST, test_jsonl)

print("train:", len(train_jsonl), TRAIN_MANIFEST)
print("val:", len(val_jsonl), VAL_MANIFEST)
print("test:", len(test_jsonl), TEST_MANIFEST)
print("train ref_audio ratio:", sum("ref_audio" in row for row in train_jsonl), "/", len(train_jsonl))

train: 461 /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/splits/train.jsonl
val: 57 /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/splits/val.jsonl
test: 57 /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/splits/test.jsonl
train ref_audio ratio: 180 / 461


## 8. LoRA Config Üret

In [18]:
import yaml

if vram_gb < 20:
    batch_size = 2
    grad_accum_steps = 8
    max_batch_tokens = 3072
elif vram_gb < 30:
    batch_size = 4
    grad_accum_steps = 4
    max_batch_tokens = 4096
else:
    batch_size = 8
    grad_accum_steps = 2
    max_batch_tokens = 8192

train_minutes = sum(record["duration"] for record in train_records) / 60
RUN_STEPS = None

if RUN_STEPS is None:
    if train_minutes < 30:
        rough_steps = 300
    elif train_minutes < 60:
        rough_steps = 500
    elif train_minutes < 120:
        rough_steps = 800
    else:
        rough_steps = 1000
else:
    rough_steps = int(RUN_STEPS)

config = {
    "pretrained_path": str(MODEL_DIR),
    "train_manifest": str(TRAIN_MANIFEST),
    "val_manifest": str(VAL_MANIFEST),
    "sample_rate": 16000,
    "out_sample_rate": 48000,
    "batch_size": batch_size,
    "grad_accum_steps": grad_accum_steps,
    "num_workers": 2,
    "num_iters": rough_steps,
    "log_interval": 10,
    "valid_interval": max(100, rough_steps // 2),
    "save_interval": max(100, rough_steps // 2),
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "warmup_steps": min(100, max(20, rough_steps // 10)),
    "max_steps": rough_steps,
    "max_batch_tokens": max_batch_tokens,
    "save_path": str(CKPT_DIR),
    "tensorboard": str(LOG_DIR),
    "lambdas": {"loss/diff": 1.0, "loss/stop": 1.0},
    "lora": {
        "enable_lm": True,
        "enable_dit": True,
        "enable_proj": False,
        "r": 32,
        "alpha": 32,
        "dropout": 0.0,
    },
}

CONFIG_PATH = CONF_DIR / "voxcpm2_lora.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

print(f"Train minutes: {train_minutes:.1f}")
print("CONFIG_PATH:", CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8"))

Train minutes: 21.3
CONFIG_PATH: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/conf/voxcpm2_lora.yaml
pretrained_path: /content/drive/MyDrive/nqai-voice/models/VoxCPM2
train_manifest: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/splits/train.jsonl
val_manifest: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/splits/val.jsonl
sample_rate: 16000
out_sample_rate: 48000
batch_size: 8
grad_accum_steps: 2
num_workers: 2
num_iters: 300
log_interval: 10
valid_interval: 150
save_interval: 150
learning_rate: 0.0001
weight_decay: 0.01
warmup_steps: 30
max_steps: 300
max_batch_tokens: 8192
save_path: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/checkpoints/lora
tensorboard: /content/drive/MyDrive/nqai-voice/finetune/neeko-proto-v0/logs/lora
lambdas:
  loss/diff: 1.0
  loss/stop: 1.0
lora:
  enable_lm: true
  enable_dit: true
  enable_proj: false
  r: 32
  alpha: 32
  dropout: 0.0



## 9. Eğitimi Başlat

In [ ]:
%cd /content/VoxCPM
!python scripts/train_voxcpm_finetune.py --config_path "{CONFIG_PATH}"

/content/VoxCPM
Detected architecture: voxcpm2 -> VoxCPM2Model
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Loading AudioVAE from pytorch: /content/drive/MyDrive/nqai-voice/models/VoxCPM2/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /content/drive/MyDrive/nqai-voice/models/VoxCPM2/model.safetensors
Generating train split: 461 examples [00:00, 101627.99 examples/s]
Generating validation split: 57 examples [00:00, 17931.10 examples/s]
Map: 100% 461/461 [00:00<00:00, 12197.67 examples/s]
Map: 100% 57/57 [00:00<00:00, 6068.52 examples/s]
base_lm.embed_tokens.weight False
base_lm.layers.0.self_attn.q_proj.weight False
base_lm.layers.0.self_attn.q_proj.lora_A True
base_lm.layers.0.self_attn.q_proj.lora_B True
base_lm.layers.0.self_attn.k_proj.weight False
base_lm.layers

## 10. Checkpoint Kontrolü

In [ ]:
print("exists:", CKPT_DIR.exists())
for path in sorted(CKPT_DIR.rglob("*")):
    print(path)

## 11. Checkpoint Inference

In [ ]:
import json
import sys
import time
from pathlib import Path

import soundfile as sf
from IPython.display import Audio, Markdown, display

for candidate in [VOX_REPO / "src", VOX_REPO]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from voxcpm import VoxCPM

try:
    from voxcpm.model.voxcpm import LoRAConfig
except Exception:
    from voxcpm.model.voxcpm2 import LoRAConfig

CHECKPOINT_NAME = "latest"
LORA_CKPT = CKPT_DIR / CHECKPOINT_NAME
assert LORA_CKPT.exists(), f"LoRA checkpoint yok: {LORA_CKPT}"

lora_config_path = LORA_CKPT / "lora_config.json"
lora_config_raw = json.loads(lora_config_path.read_text(encoding="utf-8"))
lora_config_payload = lora_config_raw.get("lora_config", lora_config_raw)
lora_config = LoRAConfig(**lora_config_payload)
print("Loading LoRA config:", lora_config.model_dump() if hasattr(lora_config, "model_dump") else lora_config)

model = VoxCPM.from_pretrained(
    str(MODEL_DIR),
    lora_config=lora_config,
    lora_weights_path=str(LORA_CKPT),
    load_denoiser=False,
    optimize=False,
)

eval_texts = [
    ("01_neeko_warm", "Merhaba küçük dostum. Bugün seninle çok güzel bir oyun oynayacağız."),
    ("02_neeko_calm", "Şimdi yavaşça gözlerini kapat, derin bir nefes al ve usulca bırak."),
    ("03_neeko_excited", "Vay canına! Bunu gerçekten sen mi yaptın? Harika görünüyor!"),
    ("04_neeko_laugh", "[laughs] Hahaha, bu gerçekten çok komikti. Bunu hiç beklemiyordum."),
    ("05_neeko_whisper", "[whispers] Şimdi çok sessiz konuşmalıyız. Küçük sırrımızı saklayalım."),
    ("06_teacher", "Bu bölümde önce kavramı anlayacağız, sonra birlikte kısa bir örnek çözeceğiz."),
]

infer_dir = OUT_DIR / f"infer_{CHECKPOINT_NAME}"
infer_dir.mkdir(parents=True, exist_ok=True)

for sid, text in eval_texts:
    t0 = time.time()
    wav = model.generate(text=text, cfg_value=2.0, inference_timesteps=10)
    elapsed = time.time() - t0
    out_path = infer_dir / f"{sid}.wav"
    sf.write(str(out_path), wav, model.tts_model.sample_rate)
    display(Markdown(f"**{sid}** · {elapsed:.1f}s · {text}"))
    display(Audio(str(out_path)))

## 12. Artifact Export

In [ ]:
import datetime
import shutil

run_meta = {
    "voice_id": VOICE_ID,
    "model": "openbmb/VoxCPM2",
    "date_utc": datetime.datetime.utcnow().isoformat(),
    "manifest": str(RAW_MANIFEST),
    "train_manifest": str(TRAIN_MANIFEST),
    "val_manifest": str(VAL_MANIFEST),
    "test_manifest": str(TEST_MANIFEST),
    "config_path": str(CONFIG_PATH),
    "checkpoint_dir": str(CKPT_DIR),
    "output_dir": str(OUT_DIR),
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 2),
    "train_clips": len(train_jsonl),
    "val_clips": len(val_jsonl),
    "test_clips": len(test_jsonl),
}

meta_path = PROJECT_DIR / "run_metadata.json"
meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

outputs_zip_base = PROJECT_DIR / f"{VOICE_ID}-outputs"
checkpoint_zip_base = PROJECT_DIR / f"{VOICE_ID}-lora-latest"

for zip_path in [Path(str(outputs_zip_base) + ".zip"), Path(str(checkpoint_zip_base) + ".zip")]:
    if zip_path.exists():
        zip_path.unlink()

shutil.make_archive(str(outputs_zip_base), "zip", root_dir=str(PROJECT_DIR), base_dir="outputs")
shutil.make_archive(str(checkpoint_zip_base), "zip", root_dir=str(CKPT_DIR), base_dir="latest")

print("metadata:", meta_path)
print("outputs zip:", str(outputs_zip_base) + ".zip")
print("checkpoint zip:", str(checkpoint_zip_base) + ".zip")